In [ ]:
import numpy as np
import math

# Helper function to calculate the log of a probability, avoiding log(0)
def log(x):
    return -math.inf if x == 0 else math.log(x)

# Function to calculate the log probability of a given path and observed sequence
def get_log_prob_of_a_given_path(path: str, seq: str) -> float:
    if len(path) != len(seq):
        raise ValueError("Path and sequence must be of the same length")

    prob = 0.0
    for i in range(len(seq)):
        p1 = path[i]  # State at position i in the path
        s1 = seq[i]   # Observation at position i in the sequence
        if i == 0:
            # Start probability for the first state in the path
            prob += log(start_prob[p1])
        else:
            # Transition probability from the previous state to the current state in the path
            prob += log(trans_prob[path[i-1]][p1])
        # Emission probability for the current state and observation
        prob += log(emit_prob[p1][s1])

    # Transition to End (only possible from 'I' as in Nature Primer)
    last_state = path[-1]
    if last_state == 'I':
        prob += log(trans_prob['I']['end'])  # Transition from 'I' to End

    return prob

# Define all the Parameters as in Nature Primer
states = ['E', '5', 'I']
start_prob = {'E': 1.0, '5': 0.0, 'I': 0.0}

trans_prob = {
    'E': {'E': 0.9, '5': 0.1},  # Transition probabilities from 'E'
    '5': {'I': 1.0},             # Transition probability from '5' to 'I'
    'I': {'I': 0.9, 'end': 0.1},  # Transition probabilities from 'I' to 'I' and 'I' to 'End'
}

emit_prob = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},  # Emission probabilities for 'E'
    '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},    # Emission probabilities for '5'
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}        # Emission probabilities for 'I'
}

# Test with a given path and sequence
path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

# Calculate the log probability of the given path
log_prob = get_log_prob_of_a_given_path(path, sequence)
print("Log probability of given path:", round(log_prob, 2))

In [ ]:
import numpy as np
import math

# Helper function to calculate the log of a probability, avoiding log(0)
def log(x):
    return -math.inf if x == 0 else math.log(x)

# Function to calculate the log probability of a given path and observed sequence
def get_log_prob_of_a_given_path(path: str, seq: str) -> float:
    if len(path) != len(seq):
        raise ValueError("Path and sequence must be of the same length")

    prob = 0.0
    for i in range(len(seq)):
        p1 = path[i]  # State at position i in the path
        s1 = seq[i]   # Observation at position i in the sequence
        if i == 0:
            prob += log(start_prob[p1])
        else:
            prob += log(trans_prob[path[i-1]][p1])
        prob += log(emit_prob[p1][s1])

    # Transition to End (only possible from 'I' as in Nature Primer)
    last_state = path[-1]
    if last_state == 'I':
        prob += log(trans_prob['I']['end'])  # Transition from 'I' to End

    return prob

# Define all the Parameters as in Nature Primer
states = ['E', '5', 'I']
start_prob = {'E': 1.0, '5': 0.0, 'I': 0.0}

trans_prob = {
    'E': {'E': 0.9, '5': 0.1},  # Transition probabilities from 'E'
    '5': {'I': 1.0},             # Transition probability from '5' to 'I'
    'I': {'I': 0.9, 'end': 0.1},  # Transition probabilities from 'I' to 'I' and 'I' to 'End'
}

emit_prob = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},  # Emission probabilities for 'E'
    '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},    # Emission probabilities for '5'
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}        # Emission probabilities for 'I'
}

# Viterbi Algorithm implementation to find the most likely path
def viterbi(obs_seq):
    # Initialize Viterbi matrix and path matrix
    V = [{}]
    path = {}

    # Initialization step for the first observation
    for state in states:
        V[0][state] = log(start_prob[state]) + log(emit_prob[state][obs_seq[0]])
        path[state] = [state]

    # Recursion step for all subsequent observations
    for t in range(1, len(obs_seq)):
        V.append({})
        new_path = {}

        for curr_state in states:
            max_prob, prev_state_best = max(
                (V[t-1][prev_state] + log(trans_prob[prev_state].get(curr_state, 0)) + log(emit_prob[curr_state].get(obs_seq[t], 0)), prev_state)
                for prev_state in states
            )
            V[t][curr_state] = max_prob
            new_path[curr_state] = path[prev_state_best] + [curr_state]

        path = new_path

    # Termination step: Find the most probable final state
    max_prob, best_final_state = max(
        (V[len(obs_seq)-1][state], state) for state in states
    )

    # Return the best path and its probability
    return path[best_final_state], max_prob

# Test with a given sequence
observed_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

best_path, best_prob = viterbi(observed_sequence)

# Convert the path to a string for readability
best_path_str = ''.join(best_path)

# Print results
print(f"Most likely path: {best_path_str}")
print(f"Log probability of this path: {round(best_prob, 2)}")